<a href="https://colab.research.google.com/github/AICHUCKY/Ai-with-Chucky-Colab-Notebooks/blob/main/Minimax_H3_Image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

###

# MiniMax H3 Image — Colab T4 Studio

- **Cell 1:** Environment setup & model downloads (4B ClipProj for fast T2I and/or 32B Qwen3-VL for Edit).
- **Cell 2:** Start inference backend & live output monitor.
- **Cell 3:** Text-to-Image generation (supports fast 4B ClipProj or 32B).
- **Cell 4:** Reference image selector (1 to 3 slots).
- **Cell 5:** Image-to-Image & Multi-Reference editing (always uses 32B Qwen3-VL).

In [ ]:
#@title 1. Setup Environment & Download Models
import html, os, subprocess, sys
from IPython.display import HTML, display
from PIL import Image

dit_choice = "AtomicChat AD-Q3_K (8.73 GB, recommended for T4)" #@param ["AtomicChat AD-Q3_K (8.73 GB, recommended for T4)", "AtomicChat AD-Q4_K (11.6 GB, highest T4 quality)", "Abiray Q3_K_M (8.91 GB)"]
encoder_choice = "4B FP8 (Fast T2I, 5.2 GB) + 32B Q2_K (Edit, 8.49 GB)" #@param ["4B FP8 (Fast T2I, 5.2 GB) + 32B Q2_K (Edit, 8.49 GB)", "4B FP8 Only (Fast T2I only, 5.2 GB — no Edit)", "32B Q2_K Only (T2I & Edit, 8.49 GB, fits 12.7GB Free RAM)", "32B Q3_K_S Only (T2I & Edit, 12.0 GB, High-RAM runtime only)"]
update_repos = False #@param {type:"boolean"}

APP_DIR = "engine"
card_handle = None
def render_setup(status, pct, stage_idx, items, err=None, hw=""):
    global card_handle
    dot = "#ef4444" if err else ("#10b981" if pct >= 100 else "#f59e0b")
    rows = "".join(
        f"<div style='background:#0a0e16;border:1px solid {'rgba(16,185,129,.4)' if ok=='done' else ('#6366f1' if ok=='act' else '#1e273a')};border-radius:7px;padding:7px 9px;font-size:11px;display:flex;justify-content:space-between;align-items:center'>"
        f"<span style='color:{'#e6edf7' if ok!='wait' else '#78849e'}'>{k}</span>"
        f"<span style='color:{'#6ee7b7' if ok=='done' else ('#c4b5fd' if ok=='act' else '#52607e')};font-family:monospace'>{v}</span></div>"
        for k, ok, v in items
    )
    err_box = f"<pre style='margin:8px 0 0;padding:8px;background:#1a0d12;border:1px solid #ef4444;border-radius:7px;color:#fca5a5;font-size:11px;white-space:pre-wrap'>{html.escape(err)}</pre>" if err else ""
    h = f"""<div style="font-family:Inter,system-ui,sans-serif;background:#111724;color:#e6edf7;border:1px solid #243047;border-radius:12px;padding:12px;max-width:960px">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;flex-wrap:wrap;gap:8px">
        <div style="display:inline-flex;align-items:center;gap:7px;background:#0a0e16;border:1px solid #243047;padding:4px 10px;border-radius:999px;font-size:12px;font-weight:600">
          <span style="width:8px;height:8px;border-radius:50%;background:{dot}"></span><span>{status}</span>
        </div>
        <span style="font-size:11px;color:#8b97b2;font-family:monospace">{hw}</span>
      </div>
      <div style="height:5px;background:#0a0e16;border-radius:99px;overflow:hidden;margin-bottom:10px"><div style="height:100%;width:{pct}%;background:linear-gradient(90deg,#7c3aed,#38bdf8);transition:width .25s"></div></div>
      <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px">{rows}</div>{err_box}</div>"""
    if card_handle is None: card_handle = display(HTML(h), display_id=True)
    else: card_handle.update(HTML(h))

def run_q(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.returncode != 0: raise RuntimeError(r.stdout[-1200:] or f"Command failed: {cmd}")

dit_map = {
    "AtomicChat AD-Q3_K (8.73 GB, recommended for T4)": ("MiniMax-H3-Image-AD-Q3_K.gguf", "https://huggingface.co/AtomicChat/MiniMax-H3-Image-GGUF/resolve/main/MiniMax-H3-Image-AD-Q3_K.gguf"),
    "AtomicChat AD-Q4_K (11.6 GB, highest T4 quality)": ("MiniMax-H3-Image-AD-Q4_K.gguf", "https://huggingface.co/AtomicChat/MiniMax-H3-Image-GGUF/resolve/main/MiniMax-H3-Image-AD-Q4_K.gguf"),
    "Abiray Q3_K_M (8.91 GB)": ("MiniMax-H3-Image-Pruned-Q3_K_M.gguf", "https://huggingface.co/Abiray/MiniMax-H3-Image-GGUF/resolve/main/MiniMax-H3-Image-Pruned-Q3_K_M.gguf"),
}
ENC_4B_FILE, ENC_4B_URL = "qwen3vl_4b_fp8_scaled.safetensors", "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"
PROJ_4B_FILE, PROJ_4B_URL = "mmh3-4b-ClipProj-v3.1.safetensors", "https://huggingface.co/NicoLab28/ClipProj-MiniMax-H3/resolve/main/mmh3-4b-ClipProj-v3.1.safetensors"
ENC_32B_Q2_FILE, ENC_32B_Q2_URL = "qwen3vl-32B-MiniMax-H3-Q2_K.gguf", "https://huggingface.co/realrebelai/MiniMax-H3_GGUFs/resolve/main/qwen3vl-32B-MiniMax-H3-Q2_K.gguf"
ENC_32B_Q3_FILE, ENC_32B_Q3_URL = "Qwen3vl_32b_h3-Q3_K_S.gguf", "https://huggingface.co/Abiray/Qwen3-VL-32B-Instruct-GGUF/resolve/main/Qwen3vl_32b_h3-Q3_K_S.gguf"

enc_downloads = []
if "4B" in encoder_choice:
    enc_downloads.append((f"{APP_DIR}/models/text_encoders", ENC_4B_FILE, ENC_4B_URL))
    enc_downloads.append((f"{APP_DIR}/models/clip_projections", PROJ_4B_FILE, PROJ_4B_URL))
active_enc_file = None
if "32B Q2_K" in encoder_choice:
    active_enc_file = ENC_32B_Q2_FILE
    enc_downloads.append((f"{APP_DIR}/models/text_encoders", ENC_32B_Q2_FILE, ENC_32B_Q2_URL))
elif "32B Q3_K_S" in encoder_choice:
    active_enc_file = ENC_32B_Q3_FILE
    enc_downloads.append((f"{APP_DIR}/models/text_encoders", ENC_32B_Q3_FILE, ENC_32B_Q3_URL))

active_dit_file, dit_url = dit_map[dit_choice]
vae_file, vae_url = "minimax_h3_image_vae_fp16.safetensors", "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/vae/minimax_h3_image_vae_fp16.safetensors"
fsize = lambda p: f"{os.path.getsize(p)/(1024**3):.2f} GB" if os.path.exists(p) else "pending"
enc_label = "4B + 32B Q2_K" if ("4B" in encoder_choice and active_enc_file) else ("4B FP8 (T2I only)" if "4B" in encoder_choice else active_enc_file)

items = [["System Tools", "act", "installing"], ["Engine & Modules", "wait", "pending"], ["Dependencies", "wait", "pending"], ["Diffusion Model", "wait", active_dit_file], ["Text Encoder(s)", "wait", enc_label], ["24-ch Image VAE", "wait", vae_file]]
try:
    render_setup("Installing acceleration tools...", 8, 1, items)
    run_q(["bash", "-c", "apt-get -qq update && apt-get -qq install -y aria2"])
    run_q([sys.executable, "-m", "pip", "install", "-q", "uv"])
    items[0][1:], items[1][1:] = ["done", "ready"], ["act", "syncing"]
    render_setup("Syncing inference engine & runtime modules...", 20, 2, items)
    for url, path in [
        ("https://github.com/Comfy-Org/ComfyUI.git", APP_DIR),
        ("https://github.com/city96/ComfyUI-GGUF.git", f"{APP_DIR}/custom_nodes/gguf_loader"),
        ("https://github.com/aaalll12322/ComfyUI-MiniMaxH3-FP16Safe.git", f"{APP_DIR}/custom_nodes/fp16_safe"),
        ("https://github.com/nicolab28/ComfyUI-ClipProj.git", f"{APP_DIR}/custom_nodes/clip_proj"),
    ]:
        if not os.path.exists(path): run_q(["git", "clone", "--depth", "1", url, path])
        elif update_repos: run_q(["git", "-C", path, "pull"])
    items[1][1:], items[2][1:] = ["done", "synced"], ["act", "installing"]
    render_setup("Installing Python dependencies...", 35, 3, items)
    run_q(["uv", "pip", "install", "--system", "--python", sys.executable, "-q", "flask", "werkzeug", "websocket-client", "psutil", "-r", f"{APP_DIR}/requirements.txt", "-r", f"{APP_DIR}/custom_nodes/gguf_loader/requirements.txt"])
    items[2][1:] = ["done", "installed"]
    for d in ["diffusion_models", "text_encoders", "clip_projections", "vae"]: os.makedirs(f"{APP_DIR}/models/{d}", exist_ok=True)
    os.makedirs(f"{APP_DIR}/input", exist_ok=True)

    items[3][1:] = ["act", f"downloading {active_dit_file}"]
    render_setup(f"Downloading {active_dit_file}...", 50, 4, items)
    run_q(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--show-console-readout=false", "--summary-interval=0", "--console-log-level=error", "-d", f"{APP_DIR}/models/diffusion_models", "-o", active_dit_file, dit_url])
    items[3][1:] = ["done", f"{active_dit_file} ({fsize(os.path.join(APP_DIR, 'models/diffusion_models', active_dit_file))})"]

    for i_e, (folder, fn, url) in enumerate(enc_downloads):
        pct_e = 65 + int(20 * (i_e + 1) / max(1, len(enc_downloads)))
        items[4][1:] = ["act", f"downloading {fn}"]
        render_setup(f"Downloading {fn}...", pct_e, 4, items)
        run_q(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--show-console-readout=false", "--summary-interval=0", "--console-log-level=error", "-d", folder, "-o", fn, url])
    items[4][1:] = ["done", enc_label]

    items[5][1:] = ["act", f"downloading {vae_file}"]
    render_setup(f"Downloading {vae_file}...", 95, 4, items)
    run_q(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--show-console-readout=false", "--summary-interval=0", "--console-log-level=error", "-d", f"{APP_DIR}/models/vae", "-o", vae_file, vae_url])
    items[5][1:] = ["done", f"{vae_file} ({fsize(os.path.join(APP_DIR, 'models/vae', vae_file))})"]

    dummy_path = f"{APP_DIR}/input/vintage_yellow_taxi.png"
    if not os.path.exists(dummy_path): Image.new("RGB", (64, 64), (255, 255, 255)).save(dummy_path)
    import psutil
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
    ram = round(psutil.virtual_memory().total / (1024**3), 1)
    render_setup("Environment & Models Ready", 100, 5, items, hw=f"{gpu} • {ram} GB RAM")
except Exception as e:
    render_setup("Setup Failed", 100, 5, items, err=str(e))
    raise

In [ ]:
#@title 2. Start Inference Backend & Output Monitor
import base64, html, json, os, random, struct, subprocess, threading, time, uuid, urllib.request
import psutil
from PIL import Image
from flask import Flask, jsonify, send_from_directory, request
from werkzeug.serving import make_server
from IPython.display import HTML, display
from google.colab import output
import websocket

vram_mode = "default" #@param ["default", "--lowvram", "--novram"]
reserve_vram_gb = 2.0 #@param {type:"number"}
preview_method = "latent2rgb" #@param ["latent2rgb", "none"]
disable_cuda_malloc = True #@param {type:"boolean"}
tiled_vae = True #@param {type:"boolean"}

APP_DIR = "engine"
_GRAPH_SRC = "https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/main/Minimax%20H3%20Image.json"
ENC_4B_FILE = "qwen3vl_4b_fp8_scaled.safetensors"
PROJ_4B_FILE = "mmh3-4b-ClipProj-v3.1.safetensors"

srv_card = None
def render_srv(status, pct, eng_st, ui_st, err=None, info=""):
    global srv_card
    dot = "#ef4444" if err else ("#10b981" if pct >= 100 else "#f59e0b")
    err_box = f"<pre style='margin:8px 0 0;padding:8px;background:#1a0d12;border:1px solid #ef4444;border-radius:7px;color:#fca5a5;font-size:11px;white-space:pre-wrap'>{html.escape(err)}</pre>" if err else ""
    h = f"""<div style="font-family:Inter,system-ui,sans-serif;background:#111724;color:#e6edf7;border:1px solid #243047;border-radius:12px;padding:12px;max-width:960px">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;flex-wrap:wrap;gap:8px">
        <div style="display:inline-flex;align-items:center;gap:7px;background:#0a0e16;border:1px solid #243047;padding:4px 10px;border-radius:999px;font-size:12px;font-weight:600">
          <span style="width:8px;height:8px;border-radius:50%;background:{dot}"></span><span>{status}</span>
        </div>
        <span style="font-size:11px;color:#8b97b2;font-family:monospace">{info}</span>
      </div>
      <div style="height:5px;background:#0a0e16;border-radius:99px;overflow:hidden;margin-bottom:10px"><div style="height:100%;width:{pct}%;background:linear-gradient(90deg,#7c3aed,#38bdf8);transition:width .25s"></div></div>
      <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:8px;font-size:11px">
        <div style="background:#0a0e16;border:1px solid #1e273a;border-radius:7px;padding:8px;display:flex;justify-content:space-between"><span>Inference Engine (:8188)</span><b style="color:{'#6ee7b7' if eng_st=='Online' else '#fbbf24'}">{eng_st}</b></div>
        <div style="background:#0a0e16;border:1px solid #1e273a;border-radius:7px;padding:8px;display:flex;justify-content:space-between"><span>Output Monitor (:5000)</span><b style="color:{'#6ee7b7' if ui_st=='Online' else '#fbbf24'}">{ui_st}</b></div>
        <div style="background:#0a0e16;border:1px solid #1e273a;border-radius:7px;padding:8px;display:flex;justify-content:space-between"><span>T4 Runtime Flags</span><b style="color:#c4b5fd">reserve={reserve_vram_gb}G • tiled={tiled_vae}</b></div>
      </div>{err_box}</div>"""
    if srv_card is None: srv_card = display(HTML(h), display_id=True)
    else: srv_card.update(HTML(h))

if "web_server" in globals() and web_server is not None:
    try: web_server.shutdown()
    except Exception: pass

eng_proc = globals().get("eng_proc")
log_file = globals().get("log_file")

def start_engine(is_recycle=False):
    global eng_proc, log_file
    if eng_proc is not None and eng_proc.poll() is None:
        try:
            eng_proc.terminate()
            eng_proc.wait(timeout=5)
        except Exception:
            try: eng_proc.kill()
            except Exception: pass
    subprocess.run(["pkill", "-9", "-f", "main.py"], stderr=subprocess.DEVNULL)
    if log_file is not None and not getattr(log_file, "closed", True):
        try: log_file.close()
        except Exception: pass
    time.sleep(1)

    cmd = [
        "python", "-u", "main.py",
        "--listen", "127.0.0.1", "--port", "8188",
        "--reserve-vram", str(reserve_vram_gb),
        "--preview-method", preview_method,
        "--cache-none",
        "--disable-smart-memory",
        "--disable-pinned-memory",
    ]
    if vram_mode != "default": cmd.append(vram_mode)
    if disable_cuda_malloc: cmd.append("--disable-cuda-malloc")

    env = os.environ.copy()
    env["MALLOC_TRIM_THRESHOLD_"] = "131072"
    env["MALLOC_MMAP_THRESHOLD_"] = "131072"

    log_file = open("engine.log", "w", buffering=1)
    eng_proc = subprocess.Popen(cmd, cwd=APP_DIR, stdout=log_file, stderr=subprocess.STDOUT, env=env)
    ui_label = "Online" if is_recycle else "Waiting"
    boot_msg = "Recycling inference engine to free RAM..." if is_recycle else "Booting inference engine..."
    render_srv(boot_msg, 30, "Starting...", ui_label)

    for i in range(90):
        try:
            if urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2).status == 200:
                if is_recycle:
                    ram_tot = round(psutil.virtual_memory().total / (1024**3), 1)
                    ram_used = round((psutil.virtual_memory().total - psutil.virtual_memory().available) / (1024**3), 1)
                    render_srv("Inference Engine & Output Monitor Online", 100, "Online", "Online", info=f"RAM {ram_used}/{ram_tot} GB • Ready for Cell 3 or 5")
                return
        except Exception:
            if eng_proc.poll() is not None:
                log_file.flush()
                tail = "".join(open("engine.log", errors="replace").readlines()[-30:])
                render_srv("Engine Startup Failed", 100, "Crashed", "Stopped", err=tail)
                raise RuntimeError("Inference engine exited unexpectedly.")
            render_srv(boot_msg, min(75, 30 + i * 2), "Starting...", ui_label)
            time.sleep(2)
    render_srv("Engine Startup Timed Out", 100, "Timeout", "Stopped")
    raise TimeoutError("Inference engine startup timed out.")

start_engine(is_recycle=False)

LOCK = threading.Lock()
JOBS = {}
CURRENT_JOB = None
if "_last_output_path" not in globals(): _last_output_path = None
if "_active_refs" not in globals(): _active_refs = {}

PANEL_HTML = """<!DOCTYPE html><html><head><meta charset="utf-8"/><style>
*{box-sizing:border-box;font-family:Inter,system-ui,sans-serif}
body{margin:0;background:#0b0f17;color:#e6edf7;padding:10px}
.card{background:#111724;border:1px solid #243047;border-radius:12px;padding:12px;max-width:960px}
.top{display:flex;justify-content:space-between;align-items:center;gap:8px;flex-wrap:wrap;margin-bottom:8px}
.pill{display:inline-flex;align-items:center;gap:6px;background:#0a0e16;border:1px solid #243047;padding:4px 10px;border-radius:999px;font-size:12px;font-weight:600}
.dot{width:8px;height:8px;border-radius:50%;background:#f59e0b;animation:p 1s infinite}
.dot.ok{background:#10b981;animation:none}.dot.err{background:#ef4444;animation:none}
@keyframes p{50%{opacity:.35}}
.meta{font-size:11px;color:#8b97b2;font-family:monospace}
.btn{cursor:pointer;background:#1b2438;border:1px solid #2e3b57;color:#e6edf7;padding:4px 10px;border-radius:7px;font-size:11px;font-weight:600;text-decoration:none}
.btn:hover{border-color:#6366f1}
.prog{height:5px;background:#0a0e16;border-radius:99px;overflow:hidden;margin-bottom:10px}
.prog>div{height:100%;width:4%;background:linear-gradient(90deg,#7c3aed,#38bdf8);transition:width .25s}
.grid{display:grid;grid-template-columns:1fr 310px;gap:10px;align-items:stretch}
@media(max-width:740px){.grid{grid-template-columns:1fr}}
.view{background:#070a10;border:1px solid #1e273a;border-radius:10px;height:360px;display:flex;align-items:center;justify-content:center;overflow:hidden;padding:6px}
.view img{max-width:100%;max-height:100%;border-radius:6px;object-fit:contain}
.side{display:flex;flex-direction:column;gap:8px;height:360px}
.steps{display:grid;grid-template-columns:repeat(5,1fr);gap:4px}
.st{background:#0a0e16;border:1px solid #1e273a;border-radius:6px;padding:5px 2px;text-align:center;font-size:10px;color:#78849e}
.st.act{border-color:#6366f1;color:#e0e7ff;background:rgba(99,102,241,.15)}
.st.ok{border-color:rgba(16,185,129,.45);color:#6ee7b7;background:rgba(16,185,129,.1)}
.logs{flex:1;background:#070a10;border:1px solid #1e273a;border-radius:8px;padding:8px;overflow-y:auto;font-size:11px;line-height:1.45}
.row{padding:3px 0;border-bottom:1px solid rgba(30,39,58,.5);word-break:break-word}
.tag{font-size:9px;font-weight:700;padding:1px 5px;border-radius:4px;margin-right:6px;text-transform:uppercase}
.t-ok{background:rgba(16,185,129,.2);color:#6ee7b7}.t-step{background:rgba(168,85,247,.2);color:#d8b4fe}.t-node{background:rgba(59,130,246,.2);color:#93c5fd}.t-err{background:rgba(239,68,68,.2);color:#fca5a5}.t-info{background:#1e293b;color:#94a3b8}
</style></head><body><div class="card">
<div class="top">
  <div class="pill"><span class="dot" id="dot"></span><span id="st">Starting...</span></div>
  <div class="meta" id="meta">__META__</div>
  <div style="display:flex;gap:6px;align-items:center">
    <span class="meta" id="tm">0.0s</span>
    <button class="btn" onclick="togRaw()" id="rawB">Raw</button>
    <a class="btn" id="dl" style="display:none" download>Download PNG</a>
  </div>
</div>
<div class="prog"><div id="bar"></div></div>
<div class="grid">
  <div class="view" id="vp"><span class="meta">Waiting for preview...</span></div>
  <div class="side">
    <div class="steps" id="stp">
      <div class="st" data-s="1">Load</div><div class="st" data-s="2">Encode</div><div class="st" data-s="3">FP16</div><div class="st" data-s="4">Sample</div><div class="st" data-s="5">Decode</div>
    </div>
    <div class="logs" id="lg"></div>
  </div>
</div></div><script>
const JID="__JOB_ID__", BASE=window.location.pathname.split('/panel/')[0];
const api=(p)=>BASE+'/'+p;
let evI=0, rawI=0, raw=false;
function togRaw(){raw=!raw;document.getElementById('rawB').textContent=raw?'Clean':'Raw';document.getElementById('lg').innerHTML='';evI=0;rawI=0;poll(true);}
function add(tag,txt){const b=document.getElementById('lg'),d=document.createElement('div');d.className='row';d.innerHTML=`<span class="tag t-${tag}">${tag}</span><span></span>`;d.lastChild.textContent=txt;b.appendChild(d);b.scrollTop=b.scrollHeight;}
async function poll(once=false){
  try{
    const d=await(await fetch(api(`api/job/${JID}?ev=${evI}&raw=${rawI}`))).json();
    document.getElementById('st').textContent=d.stage;
    document.getElementById('tm').textContent=(d.elapsed||0).toFixed(1)+'s'+(d.spt?` • ${d.spt}s/it`:'');
    document.querySelectorAll('#stp .st').forEach(e=>{const s=+e.dataset.s;e.className='st'+(d.pipe>s?' ok':(d.pipe===s?' act':''));});
    if(d.max_steps&&d.step)document.getElementById('bar').style.width=Math.round((d.step/d.max_steps)*100)+'%';
    if(!raw&&d.events){d.events.forEach(x=>add(x.tag,x.text));evI+=d.events.length;}
    if(raw&&d.raw){d.raw.forEach(x=>add('info',x));rawI+=d.raw.length;}
    if(d.state==='done'&&d.output_url){
      document.getElementById('dot').className='dot ok';
      document.getElementById('bar').style.width='100%';
      document.getElementById('vp').innerHTML=`<img src="${api(d.output_url)}"/>`;
      const dl=document.getElementById('dl');dl.href=api(d.output_url+'&download=1');dl.download=d.filename;dl.style.display='inline-block';
      return;
    }
    if(d.state==='error'){document.getElementById('dot').className='dot err';return;}
    if(d.preview_b64)document.getElementById('vp').innerHTML=`<img src="data:image/jpeg;base64,${d.preview_b64}"/>`;
    if(!once)setTimeout(poll,450);
  }catch(e){if(!once)setTimeout(poll,800);}
}
poll();
</script></body></html>"""

web_app = Flask(__name__)
@web_app.route("/panel/<jid>")
def panel_route(jid):
    return PANEL_HTML.replace("__JOB_ID__", jid).replace("__META__", JOBS.get(jid, {}).get("meta", ""))

@web_app.route("/api/job/<jid>")
def api_job(jid):
    ef, rf = int(request.args.get("ev", 0)), int(request.args.get("raw", 0))
    with LOCK:
        j = JOBS.get(jid, {})
        out = {k: v for k, v in j.items() if k not in ("events", "raw")}
        out["events"] = j.get("events", [])[ef:]
        out["raw"] = j.get("raw", [])[rf:]
    return jsonify(out)

@web_app.route("/api/output/<path:fn>")
def api_out(fn):
    return send_from_directory(f"{APP_DIR}/output", fn, as_attachment=(request.args.get("download") == "1"))

web_server = make_server("127.0.0.1", 5000, web_app, threaded=True)
threading.Thread(target=web_server.serve_forever, daemon=True).start()

def _find_32b_encoder():
    cand = globals().get("active_enc_file")
    if cand and os.path.exists(os.path.join(APP_DIR, "models/text_encoders", cand)):
        return cand
    for fn in ("Qwen3vl_32b_h3-Q3_K_S.gguf", "qwen3vl-32B-MiniMax-H3-Q2_K.gguf"):
        if os.path.exists(os.path.join(APP_DIR, "models/text_encoders", fn)):
            return fn
    return None

def _has_4b_clipproj():
    return (
        os.path.exists(os.path.join(APP_DIR, "models/text_encoders", ENC_4B_FILE))
        and os.path.exists(os.path.join(APP_DIR, "models/clip_projections", PROJ_4B_FILE))
    )

def _worker(jid, wf, is_edit, steps, actual_seed, smart_free):
    global _last_output_path, CURRENT_JOB
    t0, ws, st_times = time.time(), None, []
    def ev(tag, txt):
        with LOCK: JOBS[jid]["events"].append({"tag": tag, "text": txt})
    try:
        cid = str(uuid.uuid4())
        ws = websocket.WebSocket(); ws.connect(f"ws://127.0.0.1:8188/ws?clientId={cid}"); ws.settimeout(0.4)
        lpos = os.path.getsize("engine.log") if os.path.exists("engine.log") else 0
        req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=json.dumps({"prompt": wf, "client_id": cid}).encode(), headers={"Content-Type": "application/json"})
        pid = json.loads(urllib.request.urlopen(req).read().decode())["prompt_id"]
        ev("ok", f"Queued {pid[:8]} (seed={actual_seed})")

        def flush():
            nonlocal lpos
            with open("engine.log", "r", errors="replace") as f:
                f.seek(lpos); ch = f.read(); lpos = f.tell()
            for line in ch.replace("\r", "\n").splitlines():
                s = line.strip()
                if not s: continue
                with LOCK: JOBS[jid]["raw"].append(s)
                low = s.lower()
                if "forced compute dtype -> fp16" in low: ev("ok", "FP16Safe Tensor Cores ON")
                elif "loaded completely" in low or "loaded partially" in low: ev("ok", s.replace("[INFO] ", ""))
                elif "error" in low or "abort" in low: ev("err", s)

        done, lh = False, 0.0
        while not done:
            if eng_proc.poll() is not None:
                flush(); raise RuntimeError(f"Engine crashed (code {eng_proc.poll()})")
            flush()
            with LOCK: JOBS[jid]["elapsed"] = round(time.time() - t0, 1)
            while True:
                try: op, data = ws.recv_data()
                except websocket.WebSocketTimeoutException: break
                except websocket.WebSocketConnectionClosedException:
                    time.sleep(1.0); flush()
                    if eng_proc.poll() is not None: raise RuntimeError(f"Engine crashed (code {eng_proc.poll()})")
                    ws = websocket.WebSocket(); ws.connect(f"ws://127.0.0.1:8188/ws?clientId={cid}"); ws.settimeout(0.4); break
                if op == websocket.ABNF.OPCODE_BINARY and len(data) > 8 and struct.unpack(">I", data[:4])[0] == 1:
                    with LOCK: JOBS[jid]["preview_b64"] = base64.b64encode(data[8:]).decode("ascii")
                elif op == websocket.ABNF.OPCODE_TEXT:
                    m = json.loads(data.decode()); mt, md = m.get("type"), m.get("data", {})
                    if mt == "executing" and md.get("prompt_id") == pid:
                        nid = md.get("node")
                        if nid is None: done = True; break
                        ps = 2 if nid in ("46", "17", "28") else (3 if nid == "45" else (4 if nid == "42" else (5 if nid in ("43", "44") else 1)))
                        if nid == "42": st_times = [time.time()]
                        title = wf.get(nid, {}).get("_meta", {}).get("title", nid)
                        with LOCK:
                            JOBS[jid]["pipe"] = max(JOBS[jid]["pipe"], ps)
                            JOBS[jid]["stage"] = title
                        if nid in ("1", "2", "46", "17", "28", "45", "42", "43"): ev("node", title)
                    elif mt == "progress":
                        v, mx = md.get("value", 0), md.get("max", steps)
                        st_times.append(time.time())
                        spt = round(st_times[-1] - st_times[-2], 2) if len(st_times) >= 2 else None
                        with LOCK: JOBS[jid].update({"pipe": 4, "step": v, "max_steps": mx, "spt": spt, "stage": f"Step {v}/{mx}"})
                        ev("step", f"Step {v}/{mx}" + (f" ({spt}s/it)" if spt else ""))
            if not done and time.time() - lh > 2.5:
                lh = time.time()
                if pid in json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}").read().decode()): done = True
        time.sleep(0.3); flush()
        ent = json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}").read().decode()).get(pid, {})
        if ent.get("status", {}).get("status_str") == "error": raise RuntimeError(json.dumps(ent["status"]))
        img = ent["outputs"]["44"]["images"][0]
        rel = f"{img['subfolder']}/{img['filename']}" if img.get("subfolder") else img["filename"]
        _last_output_path = os.path.join(f"{APP_DIR}/output", rel)
        el = round(time.time() - t0, 1)
        with LOCK: JOBS[jid].update({"state": "done", "pipe": 6, "step": steps, "elapsed": el, "stage": f"Done in {el}s", "output_url": f"api/output/{rel}?t={int(time.time()*1000)}", "filename": img["filename"]})
        ev("ok", f"Saved {img['filename']} ({el}s)")
    except Exception as e:
        ev("err", str(e))
        with LOCK: JOBS[jid].update({"state": "error", "stage": "Failed", "elapsed": round(time.time() - t0, 1)})
    finally:
        if smart_free and eng_proc.poll() is None:
            try:
                urllib.request.urlopen(
                    urllib.request.Request(
                        "http://127.0.0.1:8188/free",
                        data=json.dumps({"unload_models": True, "free_memory": True}).encode(),
                        headers={"Content-Type": "application/json"},
                    ),
                    timeout=5,
                )
                time.sleep(0.5)
            except Exception:
                pass
        if ws:
            try: ws.close()
            except Exception: pass
        with LOCK: CURRENT_JOB = None

def run_minimax_ui(mode, prompt, aspect_ratio="1:1 (Square)", megapixels=3.0, ref_files=None, ref_megapixels=1.0, seed=-1, steps=6, cfg=1.0, sampler_name="lcm", scheduler="simple", denoise=1.0, filename_prefix="minimax_h3", use_fp16_safe=True, smart_free_vram=True, t2i_encoder="auto"):
    global CURRENT_JOB
    with LOCK:
        if CURRENT_JOB and JOBS.get(CURRENT_JOB, {}).get("state") == "running":
            raise RuntimeError("A generation is already running.")
    is_edit = (mode == "Edit")
    enc32 = _find_32b_encoder()
    has4b = _has_4b_clipproj()
    if is_edit:
        if not enc32:
            raise RuntimeError("Edit mode requires the 32B Qwen3-VL encoder (4B ClipProj is T2I-only). Select '4B FP8 + 32B Q2_K' or '32B Q2_K Only' in Cell 1 and re-run it.")
        use_4b = False
    else:
        if t2i_encoder.startswith("4B"):
            if not has4b: raise RuntimeError("4B ClipProj encoder not found. Select a 4B option in Cell 1 and re-run it.")
            use_4b = True
        elif t2i_encoder.startswith("32B"):
            if not enc32: raise RuntimeError("32B Qwen3-VL encoder not found. Select a 32B option in Cell 1 and re-run it.")
            use_4b = False
        else:
            if has4b: use_4b = True
            elif enc32: use_4b = False
            else: raise RuntimeError("No text encoder found. Run Cell 1 first.")

    rss_gb = 0.0
    if eng_proc.poll() is None:
        try: rss_gb = psutil.Process(eng_proc.pid).memory_info().rss / (1024**3)
        except Exception: pass
    if eng_proc.poll() is not None or (smart_free_vram and rss_gb > 5.5):
        start_engine(is_recycle=True)
    if eng_proc.poll() is not None:
        tail = "".join(open("engine.log", errors="replace").readlines()[-25:]) if os.path.exists("engine.log") else ""
        raise RuntimeError(f"Engine exited (code {eng_proc.poll()}). Re-run Cell 2.\n\n{tail}")

    refs = list(ref_files or [])[:3]
    wf = json.loads(urllib.request.urlopen(_GRAPH_SRC).read().decode())
    if "active_dit_file" in globals(): wf["1"]["inputs"]["unet_name"] = active_dit_file
    if use_4b:
        wf["2"] = {
            "inputs": {"clip_name": ENC_4B_FILE, "type": "krea2", "device": "default"},
            "class_type": "CLIPLoader",
            "_meta": {"title": "Load CLIP (Qwen3-VL-4B)"}
        }
        wf["46"] = {
            "inputs": {"clip": ["2", 0], "projection": PROJ_4B_FILE},
            "class_type": "ClipProjApply",
            "_meta": {"title": "ClipProj Apply (4B -> H3)"}
        }
        for _nid, _node in wf.items():
            if _nid == "46": continue
            for _k, _v in _node.get("inputs", {}).items():
                if _v == ["2", 0]: _node["inputs"][_k] = ["46", 0]
        enc_tag = "4B-ClipProj"
    else:
        wf["2"]["inputs"]["clip_name"] = enc32
        enc_tag = "32B"

    actual_seed = random.randint(0, 2**50) if int(seed) < 0 else int(seed)
    wf["5"]["inputs"]["value"] = is_edit
    wf["42"]["inputs"].update({"seed": actual_seed, "steps": int(steps), "cfg": float(cfg), "sampler_name": sampler_name, "scheduler": scheduler, "denoise": float(denoise)})
    wf["44"]["inputs"]["filename_prefix"] = filename_prefix
    if globals().get("tiled_vae", True) and float(megapixels) > 2.5:
        wf["43"] = {
            "inputs": {"samples": ["42", 0], "vae": ["3", 0], "tile_size": 1024, "overlap": 64, "temporal_size": 4096, "temporal_overlap": 64},
            "class_type": "VAEDecodeTiled",
            "_meta": {"title": "VAE Decode (Tiled)"}
        }
    if is_edit:
        wf["22"]["inputs"]["value"] = prompt
        wf["18"]["inputs"]["image"] = refs[0]
        wf["19"]["inputs"]["resize_type.megapixels"] = float(ref_megapixels)
        if aspect_ratio == "match_picture_1":
            with Image.open(os.path.join(APP_DIR, "input", refs[0])) as _im:
                _rw, _rh = _im.size
            _sc = ((float(megapixels) * 1_000_000) / (_rw * _rh)) ** 0.5
            wf["28"]["inputs"]["width"] = max(256, int(round((_rw * _sc) / 32) * 32))
            wf["28"]["inputs"]["height"] = max(256, int(round((_rh * _sc) / 32) * 32))
        else:
            wf["10"]["inputs"]["aspect_ratio"] = aspect_ratio
            wf["10"]["inputs"]["megapixels"] = float(megapixels)
            wf["28"]["inputs"]["width"], wf["28"]["inputs"]["height"] = ["10", 0], ["10", 1]
        for i, fn in enumerate(refs[1:], start=2):
            l, m, a = str(100+i), str(200+i), str(300+i)
            wf[l] = {"inputs": {"image": fn}, "class_type": "LoadImage", "_meta": {"title": f"Load <Picture {i}>"}}
            wf[m] = {"inputs": {"resize_type": "scale total pixels", "resize_type.megapixels": float(ref_megapixels), "scale_method": "area", "input": [l, 0]}, "class_type": "ResizeImageMaskNode", "_meta": {"title": f"Resize <Picture {i}>"}}
            wf[a] = {"inputs": {"resize_type": "scale to multiple", "resize_type.multiple": 32, "scale_method": "area", "input": [m, 0]}, "class_type": "ResizeImageMaskNode", "_meta": {"title": f"Align <Picture {i}>"}}
            wf["28"]["inputs"][f"images.image_{i}"] = [a, 0]
    else:
        wf["11"]["inputs"]["value"] = prompt
        wf["10"]["inputs"]["aspect_ratio"] = aspect_ratio
        wf["10"]["inputs"]["megapixels"] = float(megapixels)
    if use_fp16_safe:
        wf["45"] = {"inputs": {"model": ["1", 0], "debug_nan": False, "profile": False}, "class_type": "MiniMaxH3FP16Safe", "_meta": {"title": "MiniMax H3 FP16 Safe"}}
        wf["42"]["inputs"]["model"] = ["45", 0]
    jid = uuid.uuid4().hex[:10]
    with LOCK:
        CURRENT_JOB = jid
        JOBS[jid] = {"state": "running", "pipe": 1, "stage": "Initializing...", "step": 0, "max_steps": int(steps), "elapsed": 0.0, "spt": None, "preview_b64": None, "output_url": None, "events": [], "raw": [], "meta": f"{mode} • {enc_tag} • {megapixels}MP • {sampler_name} • seed={actual_seed}"}
    threading.Thread(target=_worker, args=(jid, wf, is_edit, int(steps), actual_seed, smart_free_vram), daemon=True).start()
    output.serve_kernel_port_as_iframe(5000, path=f"/panel/{jid}", height="465", cache_in_notebook=False)

ram_tot = round(psutil.virtual_memory().total / (1024**3), 1)
ram_used = round((psutil.virtual_memory().total - psutil.virtual_memory().available) / (1024**3), 1)
render_srv("Inference Engine & Output Monitor Online", 100, "Online", "Online", info=f"RAM {ram_used}/{ram_tot} GB • Ready for Cell 3 or 5")

In [ ]:
#@title 3. Text-to-Image (T2I)
prompt = "A close-up editorial studio photograph of a handcrafted miniature porcelain red fox sitting upright on a weathered mossy cedar log in a quiet misty pine forest. The fox has a glossy glazed terracotta coat with crisp white ceramic chest fur, delicate hand-painted dark glass-like eyes with tiny catchlights, pointed ears, and a bushy tail curled neatly beside its front paws. Fine damp moss fibers, tiny fern fronds, and scattered fallen pine needles sit in sharp focus across the foreground bark. Warm golden morning sunbeams cut diagonally from the upper left through soft mist between distant dark evergreen trunks, casting a bright amber rim light along the fox's ears and back and soft natural shadows underneath. Shot on a medium-format camera with an 85mm prime lens at f/2.8, natural film grain, rich tonal contrast, crisp surface glaze reflections, shallow depth of field with creamy out-of-focus forest bokeh in the background." #@param {type:"string"}
t2i_encoder = "auto (use 4B if installed, else 32B)" #@param ["auto (use 4B if installed, else 32B)", "4B (Fast T2I via ClipProj)", "32B (Full Quality)"]
aspect_ratio = "1:1 (Square)" #@param ["1:1 (Square)", "2:3 (Portrait Photo)", "3:2 (Photo)", "3:4 (Portrait Standard)", "4:3 (Standard)", "9:16 (Portrait Widescreen)", "16:9 (Widescreen)", "21:9 (Ultrawide)"]
megapixels = 3.0 #@param {type:"number"}
seed = -1 #@param {type:"integer"}
steps = 6 #@param {type:"integer"}
cfg = 1.0 #@param {type:"number"}
sampler_name = "lcm" #@param ["lcm", "euler", "euler_ancestral", "dpmpp_2m", "dpmpp_sde", "heun", "uni_pc", "res_multistep"]
scheduler = "simple" #@param ["simple", "normal", "karras", "sgm_uniform", "beta", "ddim_uniform"]
denoise = 1.0 #@param {type:"number"}
filename_prefix = "minimax_h3_t2i" #@param {type:"string"}
use_fp16_safe = True #@param {type:"boolean"}
smart_free_vram = True #@param {type:"boolean"}

run_minimax_ui("T2I", prompt, aspect_ratio=aspect_ratio, megapixels=megapixels, seed=seed, steps=steps, cfg=cfg, sampler_name=sampler_name, scheduler=scheduler, denoise=denoise, filename_prefix=filename_prefix, use_fp16_safe=use_fp16_safe, smart_free_vram=smart_free_vram, t2i_encoder=t2i_encoder)

In [ ]:
#@title 4. Set Reference Images (1 to 3 Slots) for Edit Mode
import base64, io, os, shutil
from google.colab import files
from IPython.display import HTML, clear_output, display
from PIL import Image

picture_1_source = "upload_new" #@param ["upload_new", "keep_current", "use_last_generated", "path_or_filename"]
picture_1_path = "" #@param {type:"string"}
picture_2_source = "none" #@param ["none", "upload_new", "keep_current", "use_last_generated", "path_or_filename"]
picture_2_path = "" #@param {type:"string"}
picture_3_source = "none" #@param ["none", "upload_new", "keep_current", "use_last_generated", "path_or_filename"]
picture_3_path = "" #@param {type:"string"}

IN_DIR = "engine/input"
if "_active_refs" not in globals(): _active_refs = {}
new_refs, did_up = {}, False
for slot, src, cpath in [(1, picture_1_source, picture_1_path.strip()), (2, picture_2_source, picture_2_path.strip()), (3, picture_3_source, picture_3_path.strip())]:
    if src == "none": continue
    elif src == "keep_current": new_refs[slot] = _active_refs[slot]
    elif src == "use_last_generated":
        dst = f"ref_{slot}_last.png"; shutil.copy(_last_output_path, os.path.join(IN_DIR, dst)); new_refs[slot] = dst
    elif src == "path_or_filename":
        if os.path.exists(cpath):
            dst = f"ref_{slot}_{os.path.basename(cpath)}"; shutil.copy(cpath, os.path.join(IN_DIR, dst)); new_refs[slot] = dst
        else: new_refs[slot] = cpath
    elif src == "upload_new":
        print(f"Select <Picture {slot}>:"); up = files.upload(); did_up = True
        name = next(iter(up.keys())); dst = f"ref_{slot}_{name}"; shutil.move(name, os.path.join(IN_DIR, dst)); new_refs[slot] = dst
_active_refs = new_refs
if did_up: clear_output(wait=False)
cards = []
for idx, s in enumerate(sorted(_active_refs), 1):
    fn = _active_refs[s]
    with Image.open(os.path.join(IN_DIR, fn)) as im:
        w, h = im.size; t = im.convert("RGB"); t.thumbnail((220, 220)); b = io.BytesIO(); t.save(b, "JPEG", quality=85)
        b64 = base64.b64encode(b.getvalue()).decode()
    cards.append(f"<div style='text-align:center;border:1px solid #243047;border-radius:8px;padding:8px;background:#111724;color:#e6edf7;font-size:12px'><b>&lt;Picture {idx}&gt;</b><br><img src='data:image/jpeg;base64,{b64}' style='max-height:140px;margin:6px 0;border-radius:4px'/><br>{w}x{h}</div>")
display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap'>{''.join(cards)}</div>"))

In [ ]:
#@title 5. Image-to-Image & Multi-Reference Edit (Uses 32B Qwen3-VL)
prompt = "The scene in <Picture 1> is <Subject 1>, and the subject or object in <Picture 2> is <Subject 2>. Place <Subject 2> naturally into <Subject 1> while preserving the lighting, perspective, shadows, and unedited surroundings of <Subject 1>." #@param {type:"string"}
output_aspect_ratio = "match_picture_1" #@param ["match_picture_1", "1:1 (Square)", "2:3 (Portrait Photo)", "3:2 (Photo)", "3:4 (Portrait Standard)", "4:3 (Standard)", "9:16 (Portrait Widescreen)", "16:9 (Widescreen)", "21:9 (Ultrawide)"]
output_megapixels = 3.0 #@param {type:"number"}
ref_megapixels = 1.0 #@param {type:"number"}
seed = -1 #@param {type:"integer"}
steps = 6 #@param {type:"integer"}
cfg = 1.0 #@param {type:"number"}
sampler_name = "lcm" #@param ["lcm", "euler", "euler_ancestral", "dpmpp_2m", "dpmpp_sde", "heun", "uni_pc", "res_multistep"]
scheduler = "simple" #@param ["simple", "normal", "karras", "sgm_uniform", "beta", "ddim_uniform"]
denoise = 1.0 #@param {type:"number"}
filename_prefix = "minimax_h3_edit" #@param {type:"string"}
use_fp16_safe = True #@param {type:"boolean"}
smart_free_vram = True #@param {type:"boolean"}

if not globals().get("_active_refs"): raise RuntimeError("Set at least <Picture 1> in Cell 4 first.")
run_minimax_ui("Edit", prompt, aspect_ratio=output_aspect_ratio, megapixels=output_megapixels, ref_files=[_active_refs[k] for k in sorted(_active_refs)], ref_megapixels=ref_megapixels, seed=seed, steps=steps, cfg=cfg, sampler_name=sampler_name, scheduler=scheduler, denoise=denoise, filename_prefix=filename_prefix, use_fp16_safe=use_fp16_safe, smart_free_vram=smart_free_vram)